# 🏥 Sports Injury Prediction using Machine Learning

Synthetic athlete data is used to compare multiple ML models for injury-risk prediction.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, roc_auc_score
)

import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)


def generate_sports_injury_dataset(n_samples=1000):

    age = np.random.randint(18, 41, n_samples)
    training_load = np.random.uniform(3, 25, n_samples)
    previous_injuries = np.random.randint(0, 6, n_samples)

    recovery_time = np.zeros(n_samples)
    for i in range(n_samples):
        if previous_injuries[i] > 0:
            recovery_time[i] = np.random.uniform(5, 60)

    sleep_quality = np.random.uniform(4, 10, n_samples)
    nutrition_score = np.random.uniform(1, 10, n_samples)
    muscle_fatigue = np.random.uniform(1, 10, n_samples)
    joint_flexibility = np.random.uniform(1, 10, n_samples)
    hydration_level = np.random.uniform(1, 10, n_samples)
    playing_surface = np.random.randint(0, 4, n_samples)

    risk_factor = (
        0.3 * (age - 18) / 22 +
        0.5 * (training_load - 3) / 22 +
        0.7 * previous_injuries / 5 +
        0.3 * recovery_time / 60 +
        0.4 * (10 - sleep_quality) / 6 +
        0.3 * (10 - nutrition_score) / 9 +
        0.6 * muscle_fatigue / 9 +
        0.4 * (10 - joint_flexibility) / 9 +
        0.3 * (10 - hydration_level) / 9 +
        0.2 * playing_surface / 3
    )

    risk_factor = (
        (risk_factor - risk_factor.min()) /
        (risk_factor.max() - risk_factor.min())
    )

    injury_risk = (risk_factor > 0.5).astype(int)

    injury_location = np.zeros(n_samples, dtype=int)

    for i in range(n_samples):
        if injury_risk[i] == 1:

            probs = [0.1, 0.25, 0.3, 0.2, 0.15]

            if age[i] > 30:
                probs[3] += 0.1
                probs[1] += 0.05

            if playing_surface[i] >= 2:
                probs[0] += 0.15

            if muscle_fatigue[i] > 7:
                probs[2] += 0.1

            probs = np.array(probs) / np.sum(probs)

            injury_location[i] = np.random.choice(
                np.arange(1, 6), p=probs
            )

    recovery_period = np.zeros(n_samples)

    for i in range(n_samples):

        if injury_risk[i] == 1:

            base_recovery = {
                1: 14,
                2: 28,
                3: 21,
                4: 30,
                5: 10
            }

            modifier = (
                (1.0 + (age[i] - 18) / 44) *
                (1.0 - (sleep_quality[i] - 4) / 12) *
                (1.0 - (nutrition_score[i] - 1) / 18) *
                (1.0 + previous_injuries[i] / 10)
            )

            random_factor = np.random.uniform(0.7, 1.3)

            recovery_period[i] = int(
                base_recovery[injury_location[i]] *
                modifier *
                random_factor
            )

    data = pd.DataFrame({
        'age': age,
        'training_load': training_load,
        'previous_injuries': previous_injuries,
        'recovery_time': recovery_time,
        'sleep_quality': sleep_quality,
        'nutrition_score': nutrition_score,
        'muscle_fatigue': muscle_fatigue,
        'joint_flexibility': joint_flexibility,
        'hydration_level': hydration_level,
        'playing_surface': playing_surface,
        'injury_risk': injury_risk,
        'injury_location': injury_location,
        'recovery_period': recovery_period
    })

    return data

## 1. Dataset Generation, Exploration and Model Training

In [ ]:
data = generate_sports_injury_dataset(1000)

print("Dataset Preview:")
display(data.head())

print("\nDataset Information:")
display(data.describe())

print("\nTarget Variable Distribution:")
print(f"Injury Risk: {data['injury_risk'].value_counts().to_dict()}")
print(f"Injury Location: {data['injury_location'].value_counts().to_dict()}")

plt.figure(figsize=(12, 8))
sns.heatmap(data.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Feature Correlations')
plt.tight_layout()
plt.show()

X = data.drop(
    ['injury_risk', 'injury_location', 'recovery_period'],
    axis=1
)

y_risk = data['injury_risk']

rf_for_importance = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_for_importance.fit(X, y_risk)

features_importance_df = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_for_importance.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(
    x='Importance',
    y='Feature',
    data=features_importance_df
)
plt.title('Feature Importance for Injury Risk Prediction')
plt.tight_layout()
plt.show()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_risk,
    test_size=0.2,
    random_state=42,
    stratify=y_risk
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Random Forest
# Tree-based models do not require feature scaling
rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

rf_preds = rf_model.predict(X_test)
rf_probs = rf_model.predict_proba(X_test)[:, 1]

# Logistic Regression
lr_model = LogisticRegression(
    random_state=42,
    max_iter=1000
)

lr_model.fit(X_train_scaled, y_train)

lr_preds = lr_model.predict(X_test_scaled)
lr_probs = lr_model.predict_proba(X_test_scaled)[:, 1]

# Support Vector Machine
svm_model = SVC(
    probability=True,
    random_state=42
)

svm_model.fit(X_train_scaled, y_train)

svm_preds = svm_model.predict(X_test_scaled)
svm_probs = svm_model.predict_proba(X_test_scaled)[:, 1]

## 2. Model Evaluation and Comparison

In [ ]:
models = {
    "Random Forest": (rf_preds, rf_probs),
    "Logistic Regression": (lr_preds, lr_probs),
    "Support Vector Machine": (svm_preds, svm_probs)
}

results = []

for name, (predictions, probabilities) in models.items():

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, predictions),
        "Precision": precision_score(
            y_test,
            predictions,
            zero_division=0
        ),
        "Recall": recall_score(
            y_test,
            predictions,
            zero_division=0
        ),
        "F1 Score": f1_score(
            y_test,
            predictions,
            zero_division=0
        ),
        "ROC-AUC": roc_auc_score(
            y_test,
            probabilities
        )
    })

results_df = (
    pd.DataFrame(results)
    .sort_values("Accuracy", ascending=False)
    .reset_index(drop=True)
)

print("MODEL PERFORMANCE COMPARISON")

display(
    results_df.style.format({
        "Accuracy": "{:.2%}",
        "Precision": "{:.2%}",
        "Recall": "{:.2%}",
        "F1 Score": "{:.2%}",
        "ROC-AUC": "{:.2%}"
    })
)

best_model = results_df.iloc[0]

print(f"\nBest Performing Model: {best_model['Model']}")
print(f"Accuracy: {best_model['Accuracy']:.2%}")
print(f"ROC-AUC: {best_model['ROC-AUC']:.2%}")

## 3. Performance Visualizations

In [ ]:
plt.figure(figsize=(10, 6))

bars = plt.bar(
    results_df["Model"],
    results_df["Accuracy"]
)

plt.title("Machine Learning Model Accuracy Comparison")
plt.xlabel("Models")
plt.ylabel("Accuracy")
plt.ylim(0, 1)

for bar, value in zip(bars, results_df["Accuracy"]):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value + 0.01,
        f"{value:.2%}",
        ha="center"
    )

plt.xticks(rotation=10)
plt.tight_layout()
plt.show()


results_df.set_index("Model")[
    ["Accuracy", "Precision", "Recall", "F1 Score"]
].plot(
    kind="bar",
    figsize=(12, 6)
)

plt.title("Model Performance Metrics Comparison")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. Detailed Evaluation

In [ ]:
fig, axes = plt.subplots(
    1,
    3,
    figsize=(18, 5)
)

for ax, (name, (predictions, probabilities)) in zip(
    axes,
    models.items()
):

    sns.heatmap(
        confusion_matrix(y_test, predictions),
        annot=True,
        fmt="d",
        ax=ax
    )

    ax.set_title(f"{name} - Confusion Matrix")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")

plt.tight_layout()
plt.show()


plt.figure(figsize=(10, 7))

for name, (predictions, probabilities) in models.items():

    fpr, tpr, _ = roc_curve(
        y_test,
        probabilities
    )

    auc = roc_auc_score(
        y_test,
        probabilities
    )

    plt.plot(
        fpr,
        tpr,
        label=f"{name} (AUC = {auc:.3f})"
    )

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--"
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


for name, (predictions, probabilities) in models.items():

    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    print(
        classification_report(
            y_test,
            predictions,
            zero_division=0
        )
    )

## 5. Saving Trained Models

In [ ]:
# Create models directory
os.makedirs("models", exist_ok=True)

# Save trained machine learning models
joblib.dump(
    rf_model,
    "models/random_forest_model.pkl"
)

joblib.dump(
    lr_model,
    "models/logistic_regression_model.pkl"
)

joblib.dump(
    svm_model,
    "models/svm_model.pkl"
)

# Save the fitted scaler
# Required for Logistic Regression and SVM predictions
joblib.dump(
    scaler,
    "models/scaler.pkl"
)

# Save feature names for future prediction pipelines
joblib.dump(
    list(X.columns),
    "models/feature_names.pkl"
)

print("✅ All trained models saved successfully!")

print("\nSaved Files:")
print("📁 models/random_forest_model.pkl")
print("📁 models/logistic_regression_model.pkl")
print("📁 models/svm_model.pkl")
print("📁 models/scaler.pkl")
print("📁 models/feature_names.pkl")

## 6. Final Project Summary

In [ ]:
print("=" * 60)
print("PROJECT SUMMARY")
print("=" * 60)

print(f"Dataset Size: {data.shape[0]} samples")
print(f"Number of Features: {X.shape[1]}")
print(f"Models Evaluated: {len(models)}")
print(f"Best Model: {best_model['Model']}")
print(f"Best Accuracy: {best_model['Accuracy']:.2%}")
print(f"Best ROC-AUC: {best_model['ROC-AUC']:.2%}")

print("\nProject completed successfully! 🚀")